# Introduction to Large Language Models

(Pytorch nanoGPT code is from: https://github.com/karpathy/nanoGPT/blob/master/model.py)

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
import inspect


import numpy as np

### Tokenizer

TODO




### Embedding Layer


The embedding layer is essentially a trainable look-up table. It takes as input the token indices and outputs the corresponding rows - which are the *embedding vectors* for the specified tokens. Therefore it has shape ($|V|$, $d_{model}$), where $d_{model}$ is the dimension of the embedding vectors. Bellow is an example of how it looks in code:

```python
embedding.weight = [
    [e1_1, e1_2, ..., e1_d],   # embedding for token 1
    [e2_1, e2_2, ..., e2_d],   # embedding for token 2
    ...
    [en_1, en_2, ..., en_d],   # embedding for token n
]
# where n = |V| and d = d_model
```









In [ ]:
embedding = nn.Embedding(num_embeddings=10, embedding_dim=3)

token_idx  = torch.tensor([1, 3, 8])
embedded_vectors = embedding(token_idx)

print(embedded_vectors)

tensor([[ 0.6991,  1.0415,  1.3163],
        [ 0.0860,  0.0159,  0.5319],
        [-0.5066, -0.0195, -0.4712]], grad_fn=<EmbeddingBackward0>)


### Self-attention

Self-attention is a mechanism that transforms the representation of each token in a sequence by relating it to different tokens of the sequence. This new representation can then be used by the model to e.g., predict the next word of the sequence. 

For example, lets say we have the sentence "The cat sat on the mat". We can assume a simple word tokenizer and an embedding layer which result in one embedding vector for each word:  

<div align="center">

| The    | cat   | sat   | on    | the   | mat   |
|-------|-------|-------|-------|-------|-------|
| $e_1$ | $e_2$ | $e_3$ | $e_4$ | $e_5$ | $e_6$ |

</div>

If we wanted to create a model to predict the next word, we could directly just use an MLP+softmax with input one of the embedding vectors like the above and output a probability distribution over the vocabulary.

$$
\begin{array}{c}
\text{Input Tokens (one-hot)} \quad (|V|) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Embedding Layer}
\end{array}
} \\
\downarrow \\
\text{Token Embeddings} \quad (d_{\text{model}}) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{MLP (Feed-Forward Network)}
\end{array}
} \\
\downarrow \\
\text{MLP Output} \quad (|V|) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Softmax (next token probabilities)}
\end{array}
} \\
\downarrow \\
\text{Output Probabilities} \quad (|V|)
\end{array}
$$


 However, currently each embedding $e_i$ contains information only for the particular word it embedds, regardless of the rests of the sequence. So for example given $e_5$ the model would just predict the most probable word after a "the", which is certainly not "mat". Instead, with a self-attention layer $e_5$ will be transformed into $e_5' = f(e_1, e_2, e_3, e_4, e_5)$, now containing the context. 

Bellow we will describe how self-attention is computed.


### Scaled Dot-Product Attention

- X is the embedding matrix
- Assume single attention layer after the embedding layer
- sequence_length=3

Input matrix:

 $$X  = 
\begin{bmatrix}
x_1   \\
x_2   \\
x_3  
\end{bmatrix}
 \in \mathbb{R}^{3 \times \textrm{embed\_dim}} $$
 


We apply learned projection matrices:

$$
W_Q \in \mathbb{R}^{\text{embed\_dim} \times d_q}, \quad
W_K \in \mathbb{R}^{\text{embed\_dim} \times d_k}, \quad
W_V \in \mathbb{R}^{\text{embed\_dim} \times d_v}
$$

To obtain the queries, keys, and values:


                    
$$ 
Q  = X W_Q = 
\begin{bmatrix}
q_1   \\
q_2   \\
q_3  
\end{bmatrix}
 \in \mathbb{R}^{3 \times d_q }, \ 
K  = X W_K = 
\begin{bmatrix}
k_1   \\
k_2   \\
k_3  
\end{bmatrix}
 \in \mathbb{R}^{3 \times d_k }, \ 
V  = X W_V = 
\begin{bmatrix}
v_1   \\
v_2   \\
v_3  
\end{bmatrix}
 \in \mathbb{R}^{3 \times d_v }\  
$$

Attention logits matrix $QK^T$:

$$ 
QK^T \in \mathbb{R}^{3 \times 3 }  = 

\begin{bmatrix}
q_1   \\
q_2   \\
q_3  
\end{bmatrix}

\cdot 

\begin{bmatrix}
k_1^T & k_2^T & k_3^T 
\end{bmatrix}

= 
 
\begin{bmatrix}
q_1 k_1^T & q_1 k_2^T & q_1 k_3^T \\
q_2 k_1^T & q_2 k_2^T & q_2 k_3^T \\
q_3 k_1^T & q_3 k_2^T & q_3 k_3^T
\end{bmatrix}

$$


Attention weight matrix $A$: 


$$ 

A \in \mathbb{R}^{3 \times 3 }  =

\begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}
\end{bmatrix} =

\textrm{Softmax} \left( \frac{Q K^T}{\sqrt{d_k}} \right) =  

\begin{bmatrix}
\frac{\exp \left( \frac{q_1 k_1^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_1 k_j^T}{\sqrt{d_k}} \right)} & \frac{\exp \left( \frac{q_1 k_2^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_1 k_j^T}{\sqrt{d_k}} \right)} & \frac{\exp \left( \frac{q_1 k_3^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_1 k_j^T}{\sqrt{d_k}} \right)} \\
\frac{\exp \left( \frac{q_2 k_1^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_2 k_j^T}{\sqrt{d_k}} \right)} & \frac{\exp \left( \frac{q_2 k_2^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_2 k_j^T}{\sqrt{d_k}} \right)} & \frac{\exp \left( \frac{q_2 k_3^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_2 k_j^T}{\sqrt{d_k}} \right)} \\
\frac{\exp \left( \frac{q_3 k_1^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_3 k_j^T}{\sqrt{d_k}} \right)} & \frac{\exp \left( \frac{q_3 k_2^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_3 k_j^T}{\sqrt{d_k}} \right)} & \frac{\exp \left( \frac{q_3 k_3^T}{\sqrt{d_k}} \right)}{\sum_{j=1}^{3} \exp \left( \frac{q_3 k_j^T}{\sqrt{d_k}} \right)}
\end{bmatrix}

$$

Note that $a_{ij}$ is the attention weight of query $q_i$ wrt key $k_j$ and indicates the level of attention that token $i$ pays on token $j$. TODO: explain why we scale by sqrt(dk) !!!

Attention output $Z$: 


$$

Z \in \mathbb{R}^{3 \times d_v } = \textrm{Softmax} \left( \frac{Q K^T}{\sqrt{d_k}} \right) \cdot V  = 

\begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}
\end{bmatrix}

\cdot 

\begin{bmatrix}
v_1   \\
v_2   \\
v_3  
\end{bmatrix}

= 
\begin{bmatrix}
a_{11}v_1 + a_{12}v_2 + a_{13}v_3 \\
a_{21}v_1 + a_{22}v_2 + a_{23}v_3 \\
a_{31}v_1 + a_{32}v_2 + a_{33}v_3
\end{bmatrix}

$$


$Z$ is the output of the attention layer, which weights the value vectors $v_i$ based on the computed attention weights. Each $z_i$ combines information from different value vectors (corresponding to different tokens) according to the attention given to each key by each query.



In [5]:
# Define embedding vectors for tokens (3 tokens/sq_len, 4-dimensional embeddings)
embedding_dim = 4
X = np.array([[0.1, 0.2, 0.3, 0.4],   # Embedding for token 1
              [0.5, 0.6, 0.7, 0.8],   # Embedding for token 2
              [0.9, 1.0, 1.1, 1.2]])  # Embedding for token 3

# Define weight matrices for the transformations (W_Q, W_K, W_V)
W_Q = np.random.rand(embedding_dim, embedding_dim)  # Query weight matrix
W_K = np.random.rand(embedding_dim, embedding_dim)  # Key weight matrix
W_V = np.random.rand(embedding_dim, embedding_dim)  # Value weight matrix

# Compute the Q, K, V matrices by applying the transformations to the embeddings
Q = np.dot(X, W_Q)  # Query matrix (Q)
K = np.dot(X, W_K)  # Key matrix (K)
V = np.dot(X, W_V)  # Value matrix (V)

# Define the scaling factor (sqrt of key dimension)
d_k = K.shape[1] 
scaling_factor = np.sqrt(d_k)

# Compute the attention logits (Q K^T / sqrt(d_k))
logits = np.dot(Q, K.T) / scaling_factor

# Apply softmax to the logits to get attention weights
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=1, keepdims=True)

attention_weights = softmax(logits)

# Compute the output Z (weighted sum of values)
Z = np.dot(attention_weights, V)

print("Embeddings:\n", X)
print("\nQuery Matrix (Q):\n", Q)
print("\nKey Matrix (K):\n", K)
print("\nValue Matrix (V):\n", V)
print("\nAttention Logits (Q K^T / sqrt(d_k)):\n", logits)
print("\nAttention Weights (Softmax of Logits):\n", attention_weights)
print("\nOutput Z (Weighted Sum of Values):\n", Z)

Embeddings:
 [[0.1 0.2 0.3 0.4]
 [0.5 0.6 0.7 0.8]
 [0.9 1.  1.1 1.2]]

Query Matrix (Q):
 [[0.28933965 0.54452744 0.65671681 0.32082698]
 [0.76723783 1.39047162 1.6927055  0.82227401]
 [1.24513601 2.2364158  2.72869418 1.32372104]]

Key Matrix (K):
 [[0.3122791  0.20715551 0.47338718 0.16661462]
 [0.64549905 0.61868029 1.31081583 0.47003857]
 [0.978719   1.03020507 2.14824448 0.77346253]]

Value Matrix (V):
 [[0.37200991 0.2825899  0.70429773 0.48827824]
 [0.93878223 0.90087399 1.69273318 1.40055726]
 [1.50555455 1.51915809 2.68116862 2.31283627]]

Attention Logits (Q K^T / sqrt(d_k)):
 [[0.28374619 0.76764636 1.25154653]
 [0.73297207 1.98041717 3.22786226]
 [1.18219796 3.19318798 5.204178  ]]

Attention Weights (Softmax of Logits):
 [[0.19031169 0.30875972 0.50092859]
 [0.06023429 0.20970188 0.73006384]
 [0.0155564  0.11621737 0.86822623]]

Output Z (Weighted Sum of Values):
 [[1.11483129 1.09292348 1.99975792 1.68392653]
 [1.31842308 1.31501895 2.35481644 2.0116287 ]
 [1.42205189 1.

### Masked (or causal) self-attention

In practice, transformers use a version of self-attention called masked or causal self-attention. In contrast to (bidirectional) self-attention that computes attentions scores for all tokens in the sequence, masked self-attention uses a *causal mask* that hides future tokens, so each token can attend to itself and earlier tokens. 

Example of a casual mask:

$$
\begin{array}{c|cccccc}
    & \text{The} & \text{cat} & \text{sat} & \text{on} & \text{the} & \text{mat} \\
\hline
\text{The} & 1 & 0 & 0 & 0 & 0 & 0 \\
\text{cat} & 1 & 1 & 0 & 0 & 0 & 0 \\
\text{sat} & 1 & 1 & 1 & 0 & 0 & 0 \\
\text{on}  & 1 & 1 & 1 & 1 & 0 & 0 \\
\text{the} & 1 & 1 & 1 & 1 & 1 & 0 \\
\text{mat}& 1 & 1 & 1 & 1 & 1 & 1
\end{array}
$$

Where 1 means the token can attend, 0 means it’s masked out. Note that for simplicity each word represents a token. So,
- "The" can attend only to itself.
- "cat" can attend only to "The" and "cat".
- "mat" can attend to everything in this sentence.

Usually this masking is done by setting the $q_i k_j^T$ with $i < j$ to $-\infty$ in the $QK^T$ attention logits matrix so that the application of the softmax will give zero attention weights to those positions.






In [ ]:
# Define embedding vectors for tokens (3 tokens/sq_len, 4-dimensional embeddings)
embedding_dim = 4
X = np.array([[0.1, 0.2, 0.3, 0.4],   # Embedding for token 1
              [0.5, 0.6, 0.7, 0.8],   # Embedding for token 2
              [0.9, 1.0, 1.1, 1.2]])  # Embedding for token 3

# Define weight matrices for the transformations (W_Q, W_K, W_V)
W_Q = np.random.rand(embedding_dim, embedding_dim)  # Query weight matrix
W_K = np.random.rand(embedding_dim, embedding_dim)  # Key weight matrix
W_V = np.random.rand(embedding_dim, embedding_dim)  # Value weight matrix

# Compute the Q, K, V matrices by applying the transformations to the embeddings
Q = np.dot(X, W_Q)  # Query matrix (Q)
K = np.dot(X, W_K)  # Key matrix (K)
V = np.dot(X, W_V)  # Value matrix (V)

# Define the scaling factor (sqrt of key dimension)
d_k = K.shape[1] 
scaling_factor = np.sqrt(d_k)

# Compute the attention logits (Q K^T / sqrt(d_k))
logits = np.dot(Q, K.T) / scaling_factor

# Create causal mask: shape (seq_len, seq_len)
seq_len = X.shape[0]
mask = np.tril(np.ones((seq_len, seq_len)))  # Lower triangular matrix including diagonal

# Apply mask: set logits where mask==0 to very large negative value (simulate -inf)
logits_masked = np.where(mask == 1, logits, -1e9)

# Apply softmax to the masked logits to get attention weights
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # for numerical stability
    return e_x / np.sum(e_x, axis=1, keepdims=True)

attention_weights = softmax(logits_masked)

# Compute the output Z (weighted sum of values)
Z = np.dot(attention_weights, V)

print("Embeddings:\n", X)
print("\nQuery Matrix (Q):\n", Q)
print("\nKey Matrix (K):\n", K)
print("\nValue Matrix (V):\n", V)
print("\nAttention Logits (Q K^T / sqrt(d_k)):\n", logits)
print("\nMask (1=keep, 0=mask):\n", mask)
print("\nMasked Attention Logits:\n", logits_masked)
print("\nAttention Weights (Softmax of Masked Logits):\n", attention_weights)
print("\nOutput Z (Weighted Sum of Values):\n", Z)

Embeddings:
 [[0.1 0.2 0.3 0.4]
 [0.5 0.6 0.7 0.8]
 [0.9 1.  1.1 1.2]]

Query Matrix (Q):
 [[0.37445206 0.63808738 0.53527762 0.42639362]
 [1.14268039 1.49126915 1.42375564 1.17547253]
 [1.91090873 2.34445093 2.31223366 1.92455144]]

Key Matrix (K):
 [[0.50926387 0.28276339 0.26819811 0.54880954]
 [1.13143172 0.68208795 0.66311917 1.50720176]
 [1.75359957 1.08141252 1.05804023 2.46559398]]

Value Matrix (V):
 [[0.27664098 0.61460246 0.8316572  0.46256693]
 [0.64791141 1.62860363 2.19496265 1.18161921]
 [1.01918184 2.6426048  3.55826809 1.9006715 ]]

Attention Logits (Q K^T / sqrt(d_k)):
 [[0.37434599 0.92825636 1.48216673]
 [1.01528063 2.51291775 4.01055486]
 [1.65621527 4.09757913 6.538943  ]]

Mask (1=keep, 0=mask):
 [[1. 0. 0.]
 [1. 1. 0.]
 [1. 1. 1.]]

Masked Attention Logits:
 [[ 3.74345993e-01 -1.00000000e+09 -1.00000000e+09]
 [ 1.01528063e+00  2.51291775e+00 -1.00000000e+09]
 [ 1.65621527e+00  4.09757913e+00  6.53894300e+00]]

Attention Weights (Softmax of Masked Logits):
 [[1. 

### Multi-Head Attention:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O
\quad \text{where} \quad \\
\text{head}_i = \text{Attention}(Q W_i^Q, \; K W_i^K, \; V W_i^V)
$$

where the projection matrices are:
$
\quad
W_i^Q \in \mathbb{R}^{d_{\text{model}} \times d_k}, \quad
W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}, \quad
W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}, \quad
\text{and} \quad
W^O \in \mathbb{R}^{h d_v \times d_{\text{model}}}
$


TODO: Explain more ....



In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # ! n_embd = d_model, c_attn acts as all three W_q, W_k, W_v at once and all have output dim n_embd.
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

### LayerNorm 
(https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

Given input vector $x \in \mathbb{R}^d$ (e.g., the hidden state of one token), compute:

$$\mu = \frac{1}{d} \sum_{i=1}^{d} x_i,  \  \ \sigma^2 = \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2$$

Then Layer Normalization is applied as:

$$  \textrm{LayerNorm}(x)_i =   \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma_i + \beta_i, $$

Where:

- $\gamma \in \mathbb{R}^d$: learnable scale

- $\beta \in \mathbb{R}^d$: learnable bias

Given an input $X \in \mathbb{R}^{B \times T \times d_{\textrm{model}}}$, LayerNorm($X$) will be applied independently for each token's hidden state, $x_{b,t} \in \mathbb{R}^{d_{\textrm{model}}}$, unlike BatchNorm which normalizes across a batch. Note that $\gamma$ and $\beta$ dimensions will remain $d_{\textrm{model}}$ as they are shared across tokens for all sequences.

In [7]:
class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

### MLP Layer

TODO: Explain why we need an MLP layer, also expain residual connections, GELU


#### MLP diagram:
$$
\begin{array}{c}
\text{Input } x \quad (B \times T \times d_{\text{model}}) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Linear } (d_{\text{model}} \rightarrow 4 d_{\text{model}}) \\
(\text{c\_fc})
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{GELU Activation}
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Linear } (4 d_{\text{model}} \rightarrow d_{\text{model}}) \\
(\text{c\_proj})
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Dropout}
\end{array}
} \\
\downarrow \\
\text{Output } x \quad (B \times T \times d_{\text{model}})
\end{array}
$$

#### Block Layer diagram:

$$
\begin{array}{c}
\text{Input } x \quad (B \times T \times d_{\text{model}}) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{LayerNorm (ln\_1)}
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Causal Self-Attention} \\
\text{(attn)}
\end{array}
} \\
\downarrow \\
\text{Residual: } x = x + \text{Attention Output} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{LayerNorm (ln\_2)}
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{MLP (diagram above)}
\end{array}
} \\
\downarrow \\
\text{Residual: } x = x + \text{MLP Output} \\
\downarrow \\
\text{Output of Block } (B \times T \times d_{\text{model}})
\end{array}
$$

In [ ]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### Decoder-only transformer (nanoGPT)

$$
\begin{array}{c}
\text{Input Tokens (indices)} \quad (B \times T) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Token Embedding (wte)}
\end{array}
} \\
+ \\
\boxed{
\begin{array}{c}
\text{Learned Positional Embedding (wpe)}
\end{array}
} \\
\downarrow \\
\text{Token Embeddings + Positional Embeddings (with Dropout)} \quad (B \times T \times d_{\text{model}}) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Block 1:} \\
x = x + \text{Causal Self-Attention}(\text{LayerNorm}(x)) \\
x = x + \text{MLP}(\text{LayerNorm}(x))
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Block 2:} \\
x = x + \text{Causal Self-Attention}(\text{LayerNorm}(x)) \\
x = x + \text{MLP}(\text{LayerNorm}(x))
\end{array}
} \\
\downarrow \\
\vdots \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Block N:} \\
x = x + \text{Causal Self-Attention}(\text{LayerNorm}(x)) \\
x = x + \text{MLP}(\text{LayerNorm}(x))
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Final LayerNorm (ln\_f)}
\end{array}
} \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Linear Projection (lm\_head)} \\
\text{(weights tied with token embeddings)}
\end{array}
} \\
\downarrow \\
\text{Output Logits} \quad (B \times T \times |V|) \\
\downarrow \\
\boxed{
\begin{array}{c}
\text{Softmax (next token probabilities)}
\end{array}
} \\
\downarrow \\
\text{Output Probabilities} \quad (B \times T \times |V|)
\end{array}
$$

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster


class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss
 
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

TODO: explain positional embeddings

TODO: explain training and generation

TODO: add example training code